In [ ]:

# # ResNet part - Data Cleaning Script 
# 
# This script:
# 1. Reads HAM10000 metadata and official test set ground truth
# 2. Stratified split into train (80%) and validation (20%) with fixed seed 42
# 3. Generates data/splits_local/train_seed42.csv, val_seed42.csv, test_school.csv
# 4. Validates all image paths exist
# 5. Provides PyTorch Dataset and transforms example
from pathlib import Path
import csv
import random
from collections import Counter
import pandas as pd
from sklearn.model_selection import train_test_split

# ==================== 1. Set local paths ====================
TRAIN_IMAGE_DIR = Path("/Users/liau/Downloads/trainset/HAM10000_images_All")
TRAIN_METADATA = Path("/Users/liau/Downloads/trainset/HAM10000_metadata.csv")

TEST_IMAGE_DIR = Path("/Users/liau/Downloads/TestSet/ISIC2018_Task3_Test_Images")
TEST_GROUNDTRUTH = Path("/Users/liau/Downloads/TestSet/ISIC2018_Task3_Test_GroundTruth.csv")

OUTPUT_DIR = Path.cwd() / "data" / "splits_local"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_ORDER = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
RANDOM_SEED = 42

print("Train image folder:", TRAIN_IMAGE_DIR)
print("Train metadata:", TRAIN_METADATA)
print("Test image folder:", TEST_IMAGE_DIR)
print("Test labels:", TEST_GROUNDTRUTH)
print("Output directory:", OUTPUT_DIR)
print("Label order:", LABEL_ORDER)

# %% [code]
# ==================== 2. Check existence ====================
def require_path(path, desc):
    if not path.exists():
        raise FileNotFoundError(f"Missing {desc}: {path}")

require_path(TRAIN_IMAGE_DIR, "train image folder")
require_path(TRAIN_METADATA, "train metadata")
require_path(TEST_IMAGE_DIR, "test image folder")
require_path(TEST_GROUNDTRUTH, "test labels")

train_jpgs = list(TRAIN_IMAGE_DIR.glob("*.jpg"))
test_jpgs = list(TEST_IMAGE_DIR.glob("*.jpg"))
print(f"Train images count: {len(train_jpgs)} (expected 10015)")
print(f"Test images count: {len(test_jpgs)} (expected 1511)")

# %% [code]
# ==================== 3. Read metadata and build label maps ====================
df_ham = pd.read_csv(TRAIN_METADATA)
df_ham = df_ham[df_ham['dx'].isin(LABEL_ORDER)].dropna(subset=['dx'])
print(f"HAM10000 valid samples: {len(df_ham)}")

assert df_ham['image_id'].is_unique, "image_id not unique"
id_to_dx = dict(zip(df_ham['image_id'], df_ham['dx']))

df_test = pd.read_csv(TEST_GROUNDTRUTH)
if 'image' in df_test.columns:
    df_test.rename(columns={'image': 'image_id'}, inplace=True)
if 'dx' not in df_test.columns:
    raise ValueError("Test CSV missing 'dx' column")
df_test = df_test[df_test['dx'].isin(LABEL_ORDER)]
test_id_to_dx = dict(zip(df_test['image_id'], df_test['dx']))
print(f"Test set valid samples: {len(df_test)}")

# %% [code]
# ==================== 4. Build image path mappings ====================
train_path_map = {p.stem: p.resolve() for p in train_jpgs}
test_path_map = {p.stem: p.resolve() for p in test_jpgs}

print(f"Train images mapped: {len(train_path_map)}")
print(f"Test images mapped: {len(test_path_map)}")

missing_train = [img_id for img_id in id_to_dx.keys() if img_id not in train_path_map]
if missing_train:
    print(f"Warning: {len(missing_train)} train images missing, e.g. {missing_train[:3]}")
    for img_id in missing_train:
        del id_to_dx[img_id]
    print(f"Remaining valid train samples: {len(id_to_dx)}")

missing_test = [img_id for img_id in test_id_to_dx.keys() if img_id not in test_path_map]
if missing_test:
    print(f"Warning: {len(missing_test)} test images missing, e.g. {missing_test[:3]}")
    for img_id in missing_test:
        del test_id_to_dx[img_id]
    print(f"Remaining valid test samples: {len(test_id_to_dx)}")

# %% [code]
# ==================== 5. Split train/val (stratified, fixed seed) ====================
image_ids = list(id_to_dx.keys())
labels = [id_to_dx[i] for i in image_ids]

train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    stratify=labels,
    random_state=RANDOM_SEED
)
print(f"Train samples: {len(train_ids)}")
print(f"Val samples: {len(val_ids)}")

train_labels = [id_to_dx[i] for i in train_ids]
val_labels = [id_to_dx[i] for i in val_ids]
print("Train label distribution:", Counter(train_labels))
print("Val label distribution:", Counter(val_labels))

# %% [code]
# ==================== 6. Generate CSV files ====================
def write_split_csv(ids, split_name, out_path, id_to_dx, path_map):
    rows = []
    for img_id in ids:
        if img_id not in path_map:
            print(f"Warning: {img_id} missing path, skipping")
            continue
        rows.append({
            "image_id": img_id,
            "dx": id_to_dx[img_id],
            "split": split_name,
            "image_path": str(path_map[img_id])
        })
    with open(out_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=["image_id", "dx", "split", "image_path"])
        writer.writeheader()
        writer.writerows(rows)
    print(f"Wrote {out_path}, {len(rows)} rows")

write_split_csv(train_ids, "train", OUTPUT_DIR / "train_seed42.csv", id_to_dx, train_path_map)
write_split_csv(val_ids, "val", OUTPUT_DIR / "val_seed42.csv", id_to_dx, train_path_map)
test_ids = list(test_id_to_dx.keys())
write_split_csv(test_ids, "test", OUTPUT_DIR / "test_school.csv", test_id_to_dx, test_path_map)

# %% [code]
# ==================== 7. Validate generated CSVs ====================
EXPECTED_ROWS = {
    "train_seed42.csv": len(train_ids),
    "val_seed42.csv": len(val_ids),
    "test_school.csv": len(test_ids)
}

def read_csv(path):
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

for filename, expected in EXPECTED_ROWS.items():
    csv_path = OUTPUT_DIR / filename
    rows = read_csv(csv_path)
    assert len(rows) == expected, f"{filename}: expected {expected} rows, got {len(rows)}"
    missing = [row['image_path'] for row in rows if not Path(row['image_path']).exists()]
    assert len(missing) == 0, f"{filename}: {len(missing)} invalid image paths"
    print(f"{filename}: validated, {len(rows)} rows, all images exist")

print("\nAll CSV files generated and validated!")

# %% [code]
# ==================== 8. Check class distribution ====================
for filename in ["train_seed42.csv", "val_seed42.csv", "test_school.csv"]:
    rows = read_csv(OUTPUT_DIR / filename)
    counts = Counter(row["dx"] for row in rows)
    print(f"\n{filename}")
    for label in LABEL_ORDER:
        print(f"  {label}: {counts[label]}")

# %% [code]
# ==================== 9. Show one example per class ====================
try:
    from PIL import Image
    from IPython.display import display

    rows = read_csv(OUTPUT_DIR / "train_seed42.csv")
    shown = set()
    for row in rows:
        label = row["dx"]
        if label in shown:
            continue
        print(label, row["image_id"])
        img = Image.open(row["image_path"]).convert("RGB")
        img.thumbnail((260, 220))
        display(img)
        shown.add(label)
        if len(shown) == len(LABEL_ORDER):
            break
except Exception as e:
    print("Display failed:", e)

# %% [code]
# ==================== 10. PyTorch Dataset / DataLoader example ====================
try:
    from PIL import Image
    import torch
    from torch.utils.data import Dataset, DataLoader
    from torchvision import transforms

    LABEL_TO_IDX = {label: idx for idx, label in enumerate(LABEL_ORDER)}
    IDX_TO_LABEL = {idx: label for label, idx in LABEL_TO_IDX.items()}

    class SkinLesionDataset(Dataset):
        def __init__(self, csv_path, transform=None):
            self.csv_path = Path(csv_path)
            self.transform = transform
            self.rows = read_csv(self.csv_path)

        def __len__(self):
            return len(self.rows)

        def __getitem__(self, idx):
            row = self.rows[idx]
            image = Image.open(row["image_path"]).convert("RGB")
            if self.transform is not None:
                image = self.transform(image)
            label = LABEL_TO_IDX[row["dx"]]
            return image, label

    def get_transforms(model_type="resnet", train=True, image_size=224):
        if model_type.lower() == "resnet":
            mean = [0.485, 0.456, 0.406]
            std = [0.229, 0.224, 0.225]
        else:
            mean = [0.5, 0.5, 0.5]
            std = [0.5, 0.5, 0.5]

        if train:
            return transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(20),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])
        else:
            return transforms.Compose([
                transforms.Resize((image_size, image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std),
            ])

    train_dataset = SkinLesionDataset(OUTPUT_DIR / "train_seed42.csv", transform=get_transforms(train=True))
    val_dataset = SkinLesionDataset(OUTPUT_DIR / "val_seed42.csv", transform=get_transforms(train=False))
    test_dataset = SkinLesionDataset(OUTPUT_DIR / "test_school.csv", transform=get_transforms(train=False))

    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    images, labels = next(iter(train_loader))
    print("Batch image tensor shape:", images.shape)
    print("Batch labels shape:", labels.shape)
    print("Datasets ready for training.")
except Exception as e:
    print("PyTorch demo failed. torch/torchvision may need installation.")
    print("Error:", e)

In [ ]:
 # Experiment A – Baseline ResNet18
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ==================== Experiment Configuration ====================
# Toggle the switches below to switch between Baseline and Improved version
config = {
    'use_augmentation': False,      # Enable data augmentation (random flip/rotation/color jitter)
    'use_class_weights': False,     # Use weighted loss for class imbalance
    'use_regularization': False,    # Enable Dropout + Weight Decay
    'experiment_name': 'Exp_A'   # Suffix for saved files, can be renamed to 'Improved'
}


print("="*50)
print(f"Experiment Name: {config['experiment_name']}")
print(f"Data Augmentation: {config['use_augmentation']}")
print(f"Class Weight: {config['use_class_weights']}")
print(f"Regularization: {config['use_regularization']}")
print("="*50)

# -------------------- 1. Set Random Seed for Reproducibility --------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# -------------------- 2. Hyperparameter Settings --------------------
BATCH_SIZE = 32
EPOCHS = 15                       # Set to 15 as defined in training log
LEARNING_RATE = 1e-4
IMG_SIZE = 224
WEIGHT_DECAY = 1e-4 if config['use_regularization'] else 0.0
DEVICE = torch.device("mps")
print(f"Using device: {DEVICE}")

# -------------------- 3. Dataset Path (Use pre-split CSV files) --------------------
DATA_SPLITS_DIR = "data/splits_local"          # Directory storing train_seed42.csv, val_seed42.csv, test_school.csv
TRAIN_CSV = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/train_seed42.csv")
VAL_CSV   = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/val_seed42.csv")
TEST_CSV  = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/test_school.csv")   # Test set CSV contains image_path column

# Label mapping: lowercase label -> class id
lesion_to_id = {
    'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3,
    'akiec': 4, 'vasc': 5, 'df': 6
}
# Uppercase label names for classification report display
label_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# -------------------- 4. Load Pre-split Datasets --------------------
def load_split_csv(csv_path):
    """Load pre-generated CSV, add label_id column, rename image_path to path"""
    df = pd.read_csv(csv_path)
    # Check if dx (original label) exists
    if 'dx' not in df.columns:
        raise ValueError(f"{csv_path} missing required column 'dx'")
    df['label_id'] = df['dx'].map(lesion_to_id)
    # Rename image_path to path if exists
    if 'image_path' in df.columns:
        df.rename(columns={'image_path': 'path'}, inplace=True)
    elif 'path' not in df.columns:
        raise ValueError(f"{csv_path} missing required column 'image_path' or 'path'")
    # Remove rows with invalid image paths
    df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
    return df

train_df = load_split_csv(TRAIN_CSV)
val_df   = load_split_csv(VAL_CSV)
test_df  = load_split_csv(TEST_CSV)

print(f"Train set: {len(train_df)}, Validation set: {len(val_df)}, Test set: {len(test_df)}")

# -------------------- 5. Image Transformation (Toggle augmentation by config) --------------------
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.225], std=[0.229, 0.224, 0.225])
])

if config['use_augmentation']:
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    print("Data augmentation enabled for training set")
else:
    train_transform = base_transform
    print("Data augmentation disabled")

val_test_transform = base_transform

# -------------------- 6. Custom Dataset Class --------------------
class ISICDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['label_id']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = ISICDataset(train_df, transform=train_transform)
val_dataset   = ISICDataset(val_df,   transform=val_test_transform)
test_dataset  = ISICDataset(test_df,  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -------------------- 7. Class Imbalance Handling --------------------
if config['use_class_weights']:
    class_counts = train_df['label_id'].value_counts().sort_index().values
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    class_weights = class_weights.to(DEVICE)
    print(f"Class weights: {class_weights}")
else:
    class_weights = None
    print("Class weights disabled")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# -------------------- 8. Build Model (Conditional Dropout) --------------------
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features

if config['use_regularization']:
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, 7)
    )
    print("Dropout(0.5) has been added")
else:
    model.fc = nn.Linear(num_ftrs, 7)

model = model.to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# -------------------- 9. Training & Validation Functions --------------------
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

# -------------------- 10. Main Training Loop --------------------
best_val_f1 = 0.0
history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
           'val_loss': [], 'val_acc': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_f1, _, _ = validate(model, val_loader, criterion, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Macro-F1: {val_f1:.4f}")
    
    scheduler.step(val_loss)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f"best_resnet18_{config['experiment_name']}.pth")
        print(f">>> Best model saved ({config['experiment_name']}) <<<")

# -------------------- 11. Test Set Evaluation (Label Mapping Corrected) --------------------
# Class order corresponding to id 0 ~ 6
id_to_name_lower = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
display_labels = [name.upper() for name in id_to_name_lower]  # Uppercase labels for display

model.load_state_dict(torch.load(f"best_resnet18_{config['experiment_name']}.pth"))
test_loss, test_acc, test_f1, test_preds, test_labels = validate(model, test_loader, criterion, DEVICE)

print("\n========== Test Set Results ==========")
print(f"Test Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f} | Macro-F1: {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=display_labels, digits=4))

# Plot Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=display_labels, yticklabels=display_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix ({config["experiment_name"]})')
plt.savefig(f'confusion_matrix_{config["experiment_name"]}.png')
plt.show()

# Plot Training Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 3, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 3, 3)
plt.plot(history['train_f1'], label='Train F1')
plt.plot(history['val_f1'], label='Val F1')
plt.legend()
plt.title('Macro-F1')
plt.tight_layout()
plt.savefig(f'training_curves_{config["experiment_name"]}.png')
plt.show()

# Save all results to text file
with open(f'results_{config["experiment_name"]}.txt', 'w') as f:
    f.write(f"Experiment: {config['experiment_name']}\n")
    f.write(f"Data Augmentation: {config['use_augmentation']}\n")
    f.write(f"Class Weights: {config['use_class_weights']}\n")
    f.write(f"Regularization: {config['use_regularization']}\n")
    f.write(f"Test Accuracy: {test_acc:.4f}\n")
    f.write(f"Test Macro-F1: {test_f1:.4f}\n")
    f.write("\nClassification Report:\n")
    f.write(classification_report(test_labels, test_preds, target_names=display_labels, digits=4))

In [ ]:
 # Experiment B – ResNet18 with Data Augmentation 
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ==================== Experiment Configuration ====================
# Toggle the switches below to switch between Baseline and Improved version
config = {
    'use_augmentation': True,      # Enable data augmentation (random flip/rotation/color jitter)
    'use_class_weights': False,     # Use weighted loss for class imbalance
    'use_regularization': False,    # Enable Dropout + Weight Decay
    'experiment_name': 'Exp_B'   # Suffix for saved files, can be renamed to 'Improved'
}


print("="*50)
print(f"Experiment Name: {config['experiment_name']}")
print(f"Data Augmentation: {config['use_augmentation']}")
print(f"Class Weight: {config['use_class_weights']}")
print(f"Regularization: {config['use_regularization']}")
print("="*50)

# -------------------- 1. Set Random Seed for Reproducibility --------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# -------------------- 2. Hyperparameter Settings --------------------
BATCH_SIZE = 32
EPOCHS = 15                       # Set to 15 as defined in training log
LEARNING_RATE = 1e-4
IMG_SIZE = 224
WEIGHT_DECAY = 1e-4 if config['use_regularization'] else 0.0
DEVICE = torch.device("mps")
print(f"Using device: {DEVICE}")

# -------------------- 3. Dataset Path (Use pre-split CSV files) --------------------
DATA_SPLITS_DIR = "data/splits_local"          # Directory storing train_seed42.csv, val_seed42.csv, test_school.csv
TRAIN_CSV = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/train_seed42.csv")
VAL_CSV   = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/val_seed42.csv")
TEST_CSV  = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/test_school.csv")   # Test set CSV contains image_path column

# Label mapping: lowercase label -> class id
lesion_to_id = {
    'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3,
    'akiec': 4, 'vasc': 5, 'df': 6
}
# Uppercase label names for classification report display
label_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# -------------------- 4. Load Pre-split Datasets --------------------
def load_split_csv(csv_path):
    """Load pre-generated CSV, add label_id column, rename image_path to path"""
    df = pd.read_csv(csv_path)
    # Check if dx (original label) exists
    if 'dx' not in df.columns:
        raise ValueError(f"{csv_path} missing required column 'dx'")
    df['label_id'] = df['dx'].map(lesion_to_id)
    # Rename image_path to path if exists
    if 'image_path' in df.columns:
        df.rename(columns={'image_path': 'path'}, inplace=True)
    elif 'path' not in df.columns:
        raise ValueError(f"{csv_path} missing required column 'image_path' or 'path'")
    # Remove rows with invalid image paths
    df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
    return df

train_df = load_split_csv(TRAIN_CSV)
val_df   = load_split_csv(VAL_CSV)
test_df  = load_split_csv(TEST_CSV)

print(f"Train set: {len(train_df)}, Validation set: {len(val_df)}, Test set: {len(test_df)}")

# -------------------- 5. Image Transformation (Toggle augmentation by config) --------------------
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.225], std=[0.229, 0.224, 0.225])
])

if config['use_augmentation']:
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    print("Data augmentation enabled for training set")
else:
    train_transform = base_transform
    print("Data augmentation disabled")

val_test_transform = base_transform

# -------------------- 6. Custom Dataset Class --------------------
class ISICDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['label_id']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = ISICDataset(train_df, transform=train_transform)
val_dataset   = ISICDataset(val_df,   transform=val_test_transform)
test_dataset  = ISICDataset(test_df,  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -------------------- 7. Class Imbalance Handling --------------------
if config['use_class_weights']:
    class_counts = train_df['label_id'].value_counts().sort_index().values
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    class_weights = class_weights.to(DEVICE)
    print(f"Class weights: {class_weights}")
else:
    class_weights = None
    print("Class weights disabled")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# -------------------- 8. Build Model (Conditional Dropout) --------------------
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features

if config['use_regularization']:
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, 7)
    )
    print("Dropout(0.5) has been added")
else:
    model.fc = nn.Linear(num_ftrs, 7)

model = model.to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# -------------------- 9. Training & Validation Functions --------------------
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

# -------------------- 10. Main Training Loop --------------------
best_val_f1 = 0.0
history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
           'val_loss': [], 'val_acc': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_f1, _, _ = validate(model, val_loader, criterion, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Macro-F1: {val_f1:.4f}")
    
    scheduler.step(val_loss)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f"best_resnet18_{config['experiment_name']}.pth")
        print(f">>> Best model saved ({config['experiment_name']}) <<<")

# -------------------- 11. Test Set Evaluation (Label Mapping Corrected) --------------------
# Class order corresponding to id 0 ~ 6
id_to_name_lower = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
display_labels = [name.upper() for name in id_to_name_lower]  # Uppercase labels for display

model.load_state_dict(torch.load(f"best_resnet18_{config['experiment_name']}.pth"))
test_loss, test_acc, test_f1, test_preds, test_labels = validate(model, test_loader, criterion, DEVICE)

print("\n========== Test Set Results ==========")
print(f"Test Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f} | Macro-F1: {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=display_labels, digits=4))

# Plot Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=display_labels, yticklabels=display_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix ({config["experiment_name"]})')
plt.savefig(f'confusion_matrix_{config["experiment_name"]}.png')
plt.show()

# Plot Training Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 3, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 3, 3)
plt.plot(history['train_f1'], label='Train F1')
plt.plot(history['val_f1'], label='Val F1')
plt.legend()
plt.title('Macro-F1')
plt.tight_layout()
plt.savefig(f'training_curves_{config["experiment_name"]}.png')
plt.show()

# Save all results to text file
with open(f'results_{config["experiment_name"]}.txt', 'w') as f:
    f.write(f"Experiment: {config['experiment_name']}\n")
    f.write(f"Data Augmentation: {config['use_augmentation']}\n")
    f.write(f"Class Weights: {config['use_class_weights']}\n")
    f.write(f"Regularization: {config['use_regularization']}\n")
    f.write(f"Test Accuracy: {test_acc:.4f}\n")
    f.write(f"Test Macro-F1: {test_f1:.4f}\n")
    f.write("\nClassification Report:\n")
    f.write(classification_report(test_labels, test_preds, target_names=display_labels, digits=4))

In [ ]:
 # Experiment C – ResNet18 with Data Augmentation & Class weighting
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# ==================== Experiment Configuration ====================
# Toggle the switches below to switch between Baseline and Improved version
config = {
    'use_augmentation': True,      # Enable data augmentation (random flip/rotation/color jitter)
    'use_class_weights': True,     # Use weighted loss for class imbalance
    'use_regularization': False,    # Enable Dropout + Weight Decay
    'experiment_name': 'Exp_C'   # Suffix for saved files, can be renamed to 'Improved'
}


print("="*50)
print(f"Experiment Name: {config['experiment_name']}")
print(f"Data Augmentation: {config['use_augmentation']}")
print(f"Class Weight: {config['use_class_weights']}")
print(f"Regularization: {config['use_regularization']}")
print("="*50)

# -------------------- 1. Set Random Seed for Reproducibility --------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

# -------------------- 2. Hyperparameter Settings --------------------
BATCH_SIZE = 32
EPOCHS = 15                       # Set to 15 as defined in training log
LEARNING_RATE = 1e-4
IMG_SIZE = 224
WEIGHT_DECAY = 1e-4 if config['use_regularization'] else 0.0
DEVICE = torch.device("mps")
print(f"Using device: {DEVICE}")

# -------------------- 3. Dataset Path (Use pre-split CSV files) --------------------
DATA_SPLITS_DIR = "data/splits_local"          # Directory storing train_seed42.csv, val_seed42.csv, test_school.csv
TRAIN_CSV = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/train_seed42.csv")
VAL_CSV   = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/val_seed42.csv")
TEST_CSV  = os.path.join(DATA_SPLITS_DIR, "/Users/liau/Desktop/resnew/data_副本/splits_local/test_school.csv")   # Test set CSV contains image_path column

# Label mapping: lowercase label -> class id
lesion_to_id = {
    'nv': 0, 'mel': 1, 'bkl': 2, 'bcc': 3,
    'akiec': 4, 'vasc': 5, 'df': 6
}
# Uppercase label names for classification report display
label_columns = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']

# -------------------- 4. Load Pre-split Datasets --------------------
def load_split_csv(csv_path):
    """Load pre-generated CSV, add label_id column, rename image_path to path"""
    df = pd.read_csv(csv_path)
    # Check if dx (original label) exists
    if 'dx' not in df.columns:
        raise ValueError(f"{csv_path} missing required column 'dx'")
    df['label_id'] = df['dx'].map(lesion_to_id)
    # Rename image_path to path if exists
    if 'image_path' in df.columns:
        df.rename(columns={'image_path': 'path'}, inplace=True)
    elif 'path' not in df.columns:
        raise ValueError(f"{csv_path} missing required column 'image_path' or 'path'")
    # Remove rows with invalid image paths
    df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
    return df

train_df = load_split_csv(TRAIN_CSV)
val_df   = load_split_csv(VAL_CSV)
test_df  = load_split_csv(TEST_CSV)

print(f"Train set: {len(train_df)}, Validation set: {len(val_df)}, Test set: {len(test_df)}")

# -------------------- 5. Image Transformation (Toggle augmentation by config) --------------------
base_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.225], std=[0.229, 0.224, 0.225])
])

if config['use_augmentation']:
    train_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    print("Data augmentation enabled for training set")
else:
    train_transform = base_transform
    print("Data augmentation disabled")

val_test_transform = base_transform

# -------------------- 6. Custom Dataset Class --------------------
class ISICDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.iloc[idx]['path']
        label = self.df.iloc[idx]['label_id']
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = ISICDataset(train_df, transform=train_transform)
val_dataset   = ISICDataset(val_df,   transform=val_test_transform)
test_dataset  = ISICDataset(test_df,  transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -------------------- 7. Class Imbalance Handling --------------------
if config['use_class_weights']:
    class_counts = train_df['label_id'].value_counts().sort_index().values
    class_weights = 1.0 / torch.tensor(class_counts, dtype=torch.float)
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    class_weights = class_weights.to(DEVICE)
    print(f"Class weights: {class_weights}")
else:
    class_weights = None
    print("Class weights disabled")

criterion = nn.CrossEntropyLoss(weight=class_weights)

# -------------------- 8. Build Model (Conditional Dropout) --------------------
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_ftrs = model.fc.in_features

if config['use_regularization']:
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(num_ftrs, 7)
    )
    print("Dropout(0.5) has been added")
else:
    model.fc = nn.Linear(num_ftrs, 7)

model = model.to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

# -------------------- 9. Training & Validation Functions --------------------
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validating"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds, average='macro')
    return epoch_loss, epoch_acc, epoch_f1, all_preds, all_labels

# -------------------- 10. Main Training Loop --------------------
best_val_f1 = 0.0
history = {'train_loss': [], 'train_acc': [], 'train_f1': [],
           'val_loss': [], 'val_acc': [], 'val_f1': []}

for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_acc, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc, val_f1, _, _ = validate(model, val_loader, criterion, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Macro-F1: {train_f1:.4f}")
    print(f"Val   Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | Macro-F1: {val_f1:.4f}")
    
    scheduler.step(val_loss)
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f"best_resnet18_{config['experiment_name']}.pth")
        print(f">>> Best model saved ({config['experiment_name']}) <<<")

# -------------------- 11. Test Set Evaluation (Label Mapping Corrected) --------------------
# Class order corresponding to id 0 ~ 6
id_to_name_lower = ['nv', 'mel', 'bkl', 'bcc', 'akiec', 'vasc', 'df']
display_labels = [name.upper() for name in id_to_name_lower]  # Uppercase labels for display

model.load_state_dict(torch.load(f"best_resnet18_{config['experiment_name']}.pth"))
test_loss, test_acc, test_f1, test_preds, test_labels = validate(model, test_loader, criterion, DEVICE)

print("\n========== Test Set Results ==========")
print(f"Test Loss: {test_loss:.4f} | Accuracy: {test_acc:.4f} | Macro-F1: {test_f1:.4f}")
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=display_labels, digits=4))

# Plot Confusion Matrix
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=display_labels, yticklabels=display_labels)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix ({config["experiment_name"]})')
plt.savefig(f'confusion_matrix_{config["experiment_name"]}.png')
plt.show()

# Plot Training Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.legend()
plt.title('Loss')

plt.subplot(1, 3, 2)
plt.plot(history['train_acc'], label='Train Acc')
plt.plot(history['val_acc'], label='Val Acc')
plt.legend()
plt.title('Accuracy')

plt.subplot(1, 3, 3)
plt.plot(history['train_f1'], label='Train F1')
plt.plot(history['val_f1'], label='Val F1')
plt.legend()
plt.title('Macro-F1')
plt.tight_layout()
plt.savefig(f'training_curves_{config["experiment_name"]}.png')
plt.show()

# Save all results to text file
with open(f'results_{config["experiment_name"]}.txt', 'w') as f:
    f.write(f"Experiment: {config['experiment_name']}\n")
    f.write(f"Data Augmentation: {config['use_augmentation']}\n")
    f.write(f"Class Weights: {config['use_class_weights']}\n")
    f.write(f"Regularization: {config['use_regularization']}\n")
    f.write(f"Test Accuracy: {test_acc:.4f}\n")
    f.write(f"Test Macro-F1: {test_f1:.4f}\n")
    f.write("\nClassification Report:\n")
    f.write(classification_report(test_labels, test_preds, target_names=display_labels, digits=4))

In [ ]:
"""
Model Interpretation: Grad-CAM Visualization and Success/Failure Analysis
for Experiment B (ResNet18 with data augmentation, no class weights).
"""

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget  # <-- added
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

# ==================== Configuration ====================
MODEL_PATH = "/Users/liau/Desktop/resnew/best_resnet18_Exp_B.pth"        # Change to your model file
TEST_CSV = "/Users/liau/Desktop/resnew/data_副本/splits_local/test_school.csv"
TEST_IMG_DIR = "//Users/liau/Downloads/TestSet/ISIC2018_Task3_Test_Images"
BATCH_SIZE = 32
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Label mappings (must match training)
lesion_to_id = {'nv':0, 'mel':1, 'bkl':2, 'bcc':3, 'akiec':4, 'vasc':5, 'df':6}
id_to_name = {0:'NV', 1:'MEL', 2:'BKL', 3:'BCC', 4:'AKIEC', 5:'VASC', 6:'DF'}

# -------------------- Dataset --------------------
class TestDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row["image_id"] + ".jpg")
        image = Image.open(img_path).convert('RGB')
        label = lesion_to_id[row["dx"]]
        if self.transform:
            image = self.transform(image)
        return image, label, row["image_id"]

# Transform (same as validation, no augmentation)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = TestDataset(TEST_CSV, TEST_IMG_DIR, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# -------------------- Model --------------------
def get_resnet18_frozen(num_classes=7):
    model = models.resnet18(weights=None)   # weights not needed since we load saved state
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

model = get_resnet18_frozen(num_classes=7)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

# -------------------- Inference --------------------
all_preds = []
all_labels = []
all_ids = []
all_probs = []   # store softmax probabilities for confidence

with torch.no_grad():
    for images, labels, img_ids in tqdm(test_loader, desc="Inference"):
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_ids.extend(img_ids)
        all_probs.extend(probs.cpu().numpy())

# Convert to numpy arrays
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# Compute correctness
correct_mask = (all_preds == all_labels)
confidences = np.max(all_probs, axis=1)

# Create a DataFrame for easy filtering
results_df = pd.DataFrame({
    'image_id': all_ids,
    'true_label': [id_to_name[l] for l in all_labels],
    'pred_label': [id_to_name[p] for p in all_preds],
    'correct': correct_mask,
    'confidence': confidences
})

# Save predictions for later use
results_df.to_csv("test_predictions_Exp_B.csv", index=False)
print("Predictions saved to test_predictions_Exp_B.csv")

# Print overall test metrics
print("\n===== Test Set Performance =====")
print(classification_report(all_labels, all_preds, target_names=list(id_to_name.values()), digits=4))

# Compute confusion matrix to identify confusing pairs
cm = confusion_matrix(all_labels, all_preds)
print("\n===== Confusion Matrix (raw counts) =====")
print(cm)

# Find most confusing pair (off-diagonal max)
max_confusion = 0
confused_pair = (None, None)
for i in range(len(id_to_name)):
    for j in range(len(id_to_name)):
        if i != j and cm[i, j] > max_confusion:
            max_confusion = cm[i, j]
            confused_pair = (id_to_name[i], id_to_name[j])
print(f"\nMost confusing pair: {confused_pair[0]} → {confused_pair[1]} (count = {max_confusion})")

# -------------------- Select Success and Failure Samples --------------------
# Success: correct predictions with high confidence (top 3)
success_df = results_df[results_df['correct'] == True].sort_values('confidence', ascending=False).head(3)
# Failure: wrong predictions with high confidence (top 3)
failure_df = results_df[results_df['correct'] == False].sort_values('confidence', ascending=False).head(3)

print("\n--- Selected Success Samples (Correct, High Confidence) ---")
print(success_df[['image_id', 'true_label', 'pred_label', 'confidence']])
print("\n--- Selected Failure Samples (Incorrect, High Confidence) ---")
print(failure_df[['image_id', 'true_label', 'pred_label', 'confidence']])

# -------------------- Grad-CAM Visualization --------------------
# Target layer: last convolutional layer of ResNet18 (layer4)
target_layers = [model.layer4[-1]]

def prepare_image_for_cam(img_path, transform):
    """Load image, apply transform, return tensor and original RGB (0-1)"""
    img_pil = Image.open(img_path).convert('RGB')
    orig_img = np.array(img_pil.resize((224, 224))) / 255.0
    input_tensor = transform(img_pil).unsqueeze(0)
    return input_tensor, orig_img

cam_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

os.makedirs("gradcam_output", exist_ok=True)

def generate_cam(image_id, true_label, pred_label, confidence, correct_flag):
    img_path = os.path.join(TEST_IMG_DIR, f"{image_id}.jpg")
    input_tensor, orig_img = prepare_image_for_cam(img_path, cam_transform)
    input_tensor = input_tensor.to(DEVICE)
    
    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = torch.argmax(output, dim=1).item()
    
    # Grad-CAM
    cam = GradCAM(model=model, target_layers=target_layers)
    targets = [ClassifierOutputTarget(pred_idx)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    grayscale_cam = grayscale_cam[0, :]
    
    # Overlay on image
    visualization = show_cam_on_image(orig_img, grayscale_cam, use_rgb=True)
    
    # Create a side-by-side figure: left = original, right = Grad-CAM
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    
    # Left: original image (resized 224x224)
    axes[0].imshow(orig_img)
    axes[0].set_title(f"Original: {image_id}", fontsize=8)
    axes[0].axis('off')
    
    # Right: Grad-CAM overlay
    axes[1].imshow(visualization)
    axes[1].set_title(f"Grad-CAM\nTrue: {true_label} | Pred: {pred_label}\nConf: {confidence:.3f} | {'✓' if correct_flag else '✗'}", fontsize=8)
    axes[1].axis('off')
    
    plt.tight_layout()
    out_path = f"gradcam_output/{image_id}_{'correct' if correct_flag else 'wrong'}_compare.png"
    plt.savefig(out_path, bbox_inches='tight', dpi=120)
    plt.close()
    print(f"Saved side-by-side comparison: {out_path}")
    
    # (Optional) Also save the original image alone if needed
    orig_out_path = f"gradcam_output/{image_id}_original.png"
    plt.imsave(orig_out_path, orig_img)

for _, row in success_df.iterrows():
    generate_cam(row['image_id'], row['true_label'], row['pred_label'],
                 row['confidence'], correct_flag=True)

for _, row in failure_df.iterrows():
    generate_cam(row['image_id'], row['true_label'], row['pred_label'],
                 row['confidence'], correct_flag=False)

print("\nAnalysis complete. Grad-CAM images saved in 'gradcam_output' folder.")